# Labs 19 and 20: Comps, calculators and chart reads

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


In [2]:
# From the slide: Calculators that leave a trace
import hashlib, json
def trace(name, value, inputs, formula):
    t = dict(calc=name, value=value, inputs=inputs,
             formula=formula)
    raw = json.dumps(t, sort_keys=True).encode()
    t["calc_id"] = hashlib.sha1(raw).hexdigest()[:12]
    return t

def ltm(fy, cur, prior):             # three facts
    v = fy["value"] + cur["value"] - prior["value"]
    ids = [f["fact_id"] for f in (fy, cur, prior)]
    return trace("ltm", v, ids, "FY + YTD_cur - YTD_prior")

def ev(price, sh, debt, cash, pref=0.0, nci=0.0):
    v = (price * sh["value"] + debt["value"]
         + pref + nci - cash["value"])
    ids = [f["fact_id"] for f in (sh, debt, cash)]
    return trace("ev", v, ids, "P*sh+debt+pref+nci-cash")

In [3]:
import pandas as pd
cf = pd.read_csv("data/reference/comps_facts_sample.csv"); px = pd.read_csv("data/reference/prices_asof_20260930.csv").set_index("ticker")
def fact(t, name, v): return {"fact_id": f"F-{t}-{name}", "value": float(v)}
rows = []
for _, r in cf.iterrows():
    t = r["ticker"]
    rev = ltm(fact(t, "fy_rev", r.fy_revenue), fact(t, "ytd_rev", r.ytd_revenue_current), fact(t, "ytd_rev_prior", r.ytd_revenue_prior))
    ebitda = ltm(fact(t, "fy_ebitda", r.fy_ebitda), fact(t, "ytd_ebitda", r.ytd_ebitda_current), fact(t, "ytd_ebitda_prior", r.ytd_ebitda_prior))
    e = ev(px.loc[t, "close"], fact(t, "shares", r.shares_out), fact(t, "debt", r.debt), fact(t, "cash", r.cash), pref=r.preferred, nci=r.nci)
    evr = trace("ev_revenue", e["value"] / rev["value"], [e["calc_id"], rev["calc_id"]], "EV / LTM revenue")
    eve = trace("ev_ebitda", e["value"] / ebitda["value"], [e["calc_id"], ebitda["calc_id"]], "EV / LTM EBITDA") if ebitda["value"] > 0 else None
    rows.append({"ticker": t, "ltm_revenue": rev["value"], "ltm_revenue_calc_id": rev["calc_id"], "ev": e["value"], "ev_calc_id": e["calc_id"],
                 "ev_revenue": evr["value"], "ev_revenue_calc_id": evr["calc_id"],
                 "ev_ebitda": eve["value"] if eve else None, "ev_ebitda_calc_id": eve["calc_id"] if eve else None})   # n.m. when EBITDA <= 0
comps = pd.DataFrame(rows)
Path("outputs/comps").mkdir(parents=True, exist_ok=True); comps.to_csv("outputs/comps/deal_a.csv", index=False)   # CS2 section 8: a calc_id beside every metric
print(comps[["ticker", "ev_revenue", "ev_revenue_calc_id", "ev_ebitda", "ev_ebitda_calc_id"]].round(2).to_string(index=False))
print(comps[["ev_revenue", "ev_ebitda"]].quantile([.25, .5, .75]).round(2))   # n.m. (negative EBITDA) excluded

ticker  ev_revenue ev_revenue_calc_id  ev_ebitda ev_ebitda_calc_id
  SMPA        3.21       3ce21ffb181b      16.26      131003b20f6e
  SMPB        2.63       68828103b594      19.25      9d430e42e87b
  SMPC        2.78       0af4e43ec171      12.77      4a2bc47981d2
  SMPD        2.93       45458a0d6ab2        NaN               NaN
  SMPE        4.01       b8efc6a7f206      19.80      ed9ac435bec4
  SMPF        3.09       2edb95cad353      19.60      bab9d47c7a52
      ev_revenue  ev_ebitda
0.25        2.81      16.26
0.50        3.01      19.25
0.75        3.18      19.60


In [4]:
# Chart vs table: a recorded chart read against the seller's table
reads = json.load(open("data/dataroom/deal_a/chart_reads.json"))
F = {(f["metric"], f["period"]): f["value"] for f in sd.seller_facts()}
for c in reads:
    gap = (c["value"] - F[(c["metric"], c["period"])]) / F[(c["metric"], c["period"])]
    print(c["chart"], f"chart {c['value']/1e6:.1f}m vs table {F[(c['metric'], c['period'])]/1e6:.1f}m, gap {gap:.1%}",
          "-> DISCREPANCY" if abs(gap) > llm.P["reconcile"]["tolerance_rel"] else "-> ok")

Revenue by fiscal year (CIM, page 4) chart 42.0m vs table 41.2m, gap 1.9% -> DISCREPANCY


**Exercise 19/20.** Trace SMPA's EV/EBITDA back to its input facts with the calc ids. Why is SMPD's EV/EBITDA left out of the quartiles?

In [5]:
# Exercise 19/20: record every calculation, then rebuild the comps table so the traces exist
TRACES, _trace = {}, trace
def trace(name, value, inputs, formula):   # wraps In [2]'s trace
    t = _trace(name, value, inputs, formula)
    TRACES[t["calc_id"]] = t; return t

rows = []
for _, r in cf.iterrows():                 # same loop as In [3], now through the recording trace
    t = r["ticker"]
    rev = ltm(fact(t, "fy_rev", r.fy_revenue), fact(t, "ytd_rev", r.ytd_revenue_current), fact(t, "ytd_rev_prior", r.ytd_revenue_prior))
    ebitda = ltm(fact(t, "fy_ebitda", r.fy_ebitda), fact(t, "ytd_ebitda", r.ytd_ebitda_current), fact(t, "ytd_ebitda_prior", r.ytd_ebitda_prior))
    e = ev(px.loc[t, "close"], fact(t, "shares", r.shares_out), fact(t, "debt", r.debt), fact(t, "cash", r.cash), pref=r.preferred, nci=r.nci)
    evr = trace("ev_revenue", e["value"] / rev["value"], [e["calc_id"], rev["calc_id"]], "EV / LTM revenue")
    eve = trace("ev_ebitda", e["value"] / ebitda["value"], [e["calc_id"], ebitda["calc_id"]], "EV / LTM EBITDA") if ebitda["value"] > 0 else None
    rows.append({"ticker": t, "ev_ebitda_calc_id": eve["calc_id"] if eve else None})
comps_traced = pd.DataFrame(rows)
print(len(TRACES), "traces recorded")

29 traces recorded


In [6]:
def walk(cid, depth=0):
    t = TRACES.get(cid)
    if t is None:                       # a fact id: a leaf
        print(" " * depth + cid); return
    print(" " * depth + f'{t["calc"]} = {t["value"]:,.2f} '
          f'[{t["formula"]}] ({cid})')
    for i in t["inputs"]: walk(i, depth + 1)
cid = comps_traced.loc[comps_traced.ticker == "SMPA", "ev_ebitda_calc_id"].iloc[0]
print(cid, "|", len(TRACES), "traces recorded")
walk(cid)                   # SMPA ev_ebitda

131003b20f6e | 29 traces recorded
ev_ebitda = 16.26 [EV / LTM EBITDA] (131003b20f6e)
 ev = 2,748,100,000.00 [P*sh+debt+pref+nci-cash] (b6630e39a635)
  F-SMPA-shares
  F-SMPA-debt
  F-SMPA-cash
 ltm = 169,000,000.00 [FY + YTD_cur - YTD_prior] (d68c65f5ec09)
  F-SMPA-fy_ebitda
  F-SMPA-ytd_ebitda
  F-SMPA-ytd_ebitda_prior


### Exercise 19/20: answer

**The trace for SMPA's EV/EBITDA (16.26):**
- `ev_ebitda = 16.26` = EV / LTM EBITDA (calc `131003b20f6e`)
  - `ev = 2,748,100,000` = price × shares + debt + preferred + NCI − cash (calc `b6630e39a635`),
    from facts `F-SMPA-shares`, `F-SMPA-debt`, `F-SMPA-cash`
  - `ltm EBITDA = 169,000,000` = FY + YTD current − YTD prior (calc `d68c65f5ec09`),
    from facts `F-SMPA-fy_ebitda`, `F-SMPA-ytd_ebitda`, `F-SMPA-ytd_ebitda_prior`

Every multiple can be walked down to the facts it was computed from, so a reviewer can check any
number in the comps table against its source.

**Why is SMPD's EV/EBITDA left out of the quartiles?** SMPD's LTM EBITDA is negative:
−4m + 1m − (−2m) = −1m. A multiple of a negative denominator isn't meaningful (it would be a
negative "valuation" that sorts below every real one), so it is marked n.m. (NaN) and excluded,
not set to zero, which would also distort the quartiles. That is why the EV/EBITDA quartiles
(16.26 / 19.25 / 19.60) use five companies, while EV/revenue uses all six.

**Where sign errors could hide:** cash is subtracted in EV, not added; a negative prior-year YTD
(SMPD's −2m) is subtracted, so it *adds* 2m; and debt reported as a negative liability would flip
the EV.

**A gap in the trace:** the share price is not a leaf in the walk (it's read straight from
`prices_asof_20260930.csv`). Adding the price snapshot row as a fact, cited to its CSV row,
would make every input traceable to a source.